In [78]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import Optional, Dict, List
import csv
import random

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [43]:
def find_appid(game_name: str, lookup_df: pd.DataFrame) -> Optional[int]:
    """
    Find AppID from the lookup file by matching game name
    """
    game_name_clean = game_name.strip()
    
    # Try exact match first (case insensitive)
    exact_match = lookup_df[lookup_df['name'].str.lower() == game_name_clean.lower()]
    if not exact_match.empty:
        return int(exact_match.iloc[0]['appid'])
    
    # Try partial match
    partial_match = lookup_df[lookup_df['name'].str.contains(game_name_clean, case=False, na=False, regex=False)]
    if not partial_match.empty:
        # Prefer closer matches
        for idx, row in partial_match.iterrows():
            if game_name_clean.lower() in row['name'].lower():
                return int(row['appid'])
        return int(partial_match.iloc[0]['appid'])
    
    return None

In [85]:
def getData(app_id):
    url = f'https://store.steampowered.com/api/appdetails?appids={app_id}&l=english'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers = headers)

    if response.status_code!= 200:
        print(f'Status Code: {response.status_code}')
        return None
    data = response.json()

    return data


In [86]:
api_results = {}
lookup_df = pd.read_csv("complete_steam_lookup_2026.csv")
with open('game_list.csv', mode = 'r', newline = '') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        app_id = find_appid(row[0], lookup_df)
        df_details = getData(app_id)
        api_results[row[0]] = df_details[str(app_id)]
        time.sleep(random.uniform(2, 5))

In [96]:
for game_name, data in api_results.items():
    print(data)

{'success': True, 'data': {'type': 'game', 'name': 'Counter-Strike 2', 'steam_appid': 730, 'required_age': 0, 'is_free': True, 'dlc': [2678630], 'detailed_description': 'For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.<br><br>A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools.<br><br>In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features:<br><br><ul class="bb_ul"><li>All-new CS Ratings with the updated Premier mode<br></li><li>Global and Regional leaderboards<br></li><li>Upgraded and overhauled maps<br></li><li>Game-changing dynam

In [ ]:
categories_selection = ['Multi-player', 'Co-op', 'Online Co-op', 'Single-player']

cluster_features = []
for game_name, data in api_results.items():
    if 'data' in data
        genres = [game['description'] for game in data['data'].get('genres', [])]
        clean_genres = [genre for genre in genres if genre != 'Free To Play']

        categories = [game['description'] for game in data['data'].get('categories', [])]
        clean_categories = [cat for cat in categories if cat in categories_selection]

        is_free = 1 if data['data'].get('is_free') else 0

        cluster_features.append({
            'Game': game_name,
            'is_free': is_free,
            'genres':clean_genres,
            'categories':clean_categories

        })
df_cluster = pd.DataFrame(cluster_features)
df_cluster = df_cluster.set_index('Game')

genre_mlb = MultiLabelBinarizer()
categories_mlb = MultiLabelBinarizer()

genres = genre_mlb.fit_transform(df_cluster['genres'])
genre_labeled = pd.DataFrame(genres, columns = genre_mlb.classes_ , index = df_cluster.index)

categories = categories_mlb.fit_transform(df_cluster['categories'])
categories_labeled = pd.DataFrame(categories, columns = categories_mlb.classes_ , index = df_cluster.index)

df_cluster = pd.concat([df_cluster[['is_free']], genre_labeled, categories_labeled], axis = 1)

df_cluster.to_csv('cluster_details.csv')

KeyError: 'data'

In [ ]:
X = df_clustering.copy()